# Decision Trees: Forest Fires (Regression) and Obesity (Classification)

This notebook has two projects. Every section has three parts:

1. **Explanation** (markdown): what the next code does and why.
2. **Code**: the step itself.
3. **Findings feedback**: some cells print an automatic verdict from *your* results, and a short note says how to read it.

| | Part A | Part B |
|---|---|---|
| Task | Regression | Classification |
| Data | Forest fires (UCI id 162) | Obesity levels (UCI id 544) |
| Target | `area` (burned hectares) | `NObeyesdad` (7 classes) |
| Model | `DecisionTreeRegressor` | `DecisionTreeClassifier` |

Run the cells from top to bottom.

## 1. Setup

**What this does:** imports the libraries and defines one hyperparameter search space.

**Key point:** `max_depth`, `min_samples_split` and `min_samples_leaf` exist in both tree types, so one dictionary serves both models. They control tree complexity: deeper trees with small leaves overfit, shallow trees with large leaves underfit.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import randint

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, f1_score, classification_report,
                             confusion_matrix)

sns.set_theme(style="whitegrid")

param_dist = {
    "max_depth": randint(2, 40),
    "min_samples_split": randint(2, 40),
    "min_samples_leaf": randint(1, 40),
}

# Part A: Forest Fires (Regression)

**Problem:** predict the burned area of a fire from its location, month, day, fire-weather indices (`FFMC`, `DMC`, `DC`, `ISI`) and weather (`temp`, `RH`, `wind`, `rain`).

## A1. Load and inspect the data

**What this does:** loads the CSV directly from UCI and checks size, types, summary statistics, missing values and duplicates.

**Why:** never model before you know what the data looks like. Wrong types, missing values and duplicates are the most common silent problems.

In [ ]:
url_fire = "https://archive.ics.uci.edu/ml/machine-learning-databases/forest-fires/forestfires.csv"
forest = pd.read_csv(url_fire)

print("Shape:", forest.shape)
print(forest.dtypes)
print("Missing values:", forest.isnull().sum().sum())
print("Duplicate rows:", forest.duplicated().sum())
forest.describe()

## A2. Remove duplicates

**What this does:** `duplicated()` flags a row only when *all* columns match an earlier row (the first copy is not flagged). `drop_duplicates()` removes the repeats.

**Why:** if a copy lands in the training set and its twin in the test set, the model is tested on data it has already seen, which inflates the score.

**Common mistake:** writing `drop_duplicates` without `()` assigns the method itself instead of the cleaned data.

In [ ]:
print(forest[forest.duplicated()])
forest = forest.drop_duplicates().reset_index(drop=True)
print("Shape after dropping duplicates:", forest.shape)

## A3. EDA: fires per month

**What this does:** counts fires per month and orders the bars from January to December.

**Why:** `value_counts()` sorts by frequency, which hides the seasonal pattern. Reindexing to calendar order shows it.

**What to look for:** which months dominate, and which have so few fires that their averages would be unreliable.

In [ ]:
order = ["jan", "feb", "mar", "apr", "may", "jun",
         "jul", "aug", "sep", "oct", "nov", "dec"]
mcount = forest["month"].value_counts().reindex(order, fill_value=0)

plt.figure(figsize=(10, 5))
plt.bar(mcount.index, mcount.values)
plt.xlabel("Month")
plt.ylabel("Count")
plt.title("Fires per Month")
plt.show()

top2 = mcount.sort_values(ascending=False).head(2)
print(f"FINDING: {top2.index[0]} and {top2.index[1]} have the most fires "
      f"({top2.iloc[0]} and {top2.iloc[1]} of {len(forest)}).")
rare = mcount[mcount < 10].index.tolist()
print("Months with fewer than 10 fires (averages unreliable):", rare)

## A4. EDA: the target distribution

**What this does:** compares the raw burned area with `log1p(area)`.

**Why a log transform:** about half the fires have `area = 0` and a few are enormous. The raw target is extremely right-skewed, so a few giant fires would dominate the squared error. `log1p` (log of 1 + x) compresses the large values and handles zeros safely.

**Note:** the reason for the transform is the skew of `area`, not the August and September peak in the month chart.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(forest["area"], bins=40, ax=ax[0])
ax[0].set_title("area (raw)")
sns.histplot(np.log1p(forest["area"]), bins=40, ax=ax[1])
ax[1].set_title("log1p(area)")
plt.show()

zero_share = (forest["area"] == 0).mean()
print(f"FINDING: {zero_share:.0%} of fires have area = 0; "
      f"skew of raw area = {forest['area'].skew():.1f}, "
      f"skew after log1p = {np.log1p(forest['area']).skew():.1f}.")

## A5. EDA: correlations

**What this does:** finds numeric feature pairs with a correlation above 0.5 in absolute value, and draws a heatmap.

**Is it required for a tree?** No. Trees split on one feature at a time and do not assume linearity, so correlated features do not break them. It is still useful for spotting redundant features (importance gets split between them), for leakage, and for explaining why the target is hard to predict.

In [ ]:
corr = forest.corr(numeric_only=True)     # numeric_only skips month/day text

pairs = []
for i in corr.columns:
    for j in corr.columns:
        if i < j and abs(corr.loc[i, j]) > 0.5:
            pairs.append((i, j, round(corr.loc[i, j], 2)))
for p in pairs:
    print(p)

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.show()

with_target = corr["area"].drop("area").abs().sort_values(ascending=False)
print(f"FINDING: strongest feature-to-area correlation is {with_target.index[0]} "
      f"({with_target.iloc[0]:.2f}). Values this small mean weak linear signal.")

## A6. Preprocessing and train/test split

**What this does:**
- Maps `month` and `day` to numbers (trees need numeric input).
- Creates `log_area` as a *new* column and keeps the original `area`, so errors can be converted back to hectares later.
- Splits 80% train and 20% test with a fixed `random_state`.

In [ ]:
month_map = {m: i + 1 for i, m in enumerate(order)}
day_map = {"mon": 1, "tue": 2, "wed": 3, "thu": 4, "fri": 5, "sat": 6, "sun": 7}
forest["month"] = forest["month"].map(month_map)
forest["day"] = forest["day"].map(day_map)
forest["log_area"] = np.log1p(forest["area"])

X = forest.drop(columns=["area", "log_area"])
y = forest["log_area"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(X_train.shape, X_test.shape)

## A7. Tune the regression tree

**What this does:** `RandomizedSearchCV` tries 50 random hyperparameter combinations. Each is scored with 5-fold cross-validation (250 fits in total), and the best one is refit on all training data.

**Key points:**
- It is faster than a full grid search over a wide space.
- `scoring="neg_mean_squared_error"`: scikit-learn maximizes scores, so errors are negated (closer to 0 is better).
- `random_state` makes results repeatable.

In [ ]:
reg_search = RandomizedSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_dist, n_iter=50, cv=5,
    scoring="neg_mean_squared_error", random_state=42,
)
reg_search.fit(X_train, y_train)
print("Best params:", reg_search.best_params_)
print("Best CV MSE:", -reg_search.best_score_)

## A8. Evaluate against a baseline

**What this does:** compares the tuned tree with a `DummyRegressor` that predicts the training median for every row.

**Why:** a metric alone means nothing. If the tree cannot beat a model that ignores every feature, the features carry little usable signal.

**Metrics:**
- **MAE:** average size of the miss.
- **RMSE:** like MAE but punishes large misses more. RMSE much larger than MAE means a few predictions are very wrong.
- **R²:** share of variance explained. 1 is perfect, 0 equals predicting the mean, negative is worse than the mean.

In [ ]:
base_reg = DummyRegressor(strategy="median").fit(X_train, y_train)

rows = []
for name, m in [("baseline", base_reg), ("tree", reg_search)]:
    p = m.predict(X_test)
    rows.append({
        "model": name,
        "MAE": mean_absolute_error(y_test, p),
        "RMSE": mean_squared_error(y_test, p) ** 0.5,
        "R2": r2_score(y_test, p),
    })
reg_results = pd.DataFrame(rows).set_index("model").round(3)
reg_results

### Findings feedback: regression (automatic)

The next cell reads your own numbers and prints a verdict, so it always matches your run.

In [ ]:
base_rmse = reg_results.loc["baseline", "RMSE"]
tree_rmse = reg_results.loc["tree", "RMSE"]
tree_r2 = reg_results.loc["tree", "R2"]
gain = (base_rmse - tree_rmse) / base_rmse * 100

print(f"Tree RMSE is {gain:+.1f}% vs the baseline.")
if gain > 5:
    print("VERDICT: the tree beats the baseline clearly. Confirm with repeated CV that it is not a lucky split.")
elif gain > -5:
    print("VERDICT: the tree is about equal to the baseline. The features add little signal; "
          "this is common for forest fires (noisy target, missing fuel, terrain and human-cause data).")
else:
    print("VERDICT: the tree is worse than the baseline, a sign of overfitting. "
          "Use a smaller max_depth and a larger min_samples_leaf.")

if tree_r2 < 0:
    print(f"R2 = {tree_r2:.2f} (negative): worse than always predicting the mean.")
elif tree_r2 < 0.1:
    print(f"R2 = {tree_r2:.2f}: explains almost none of the variation.")
else:
    print(f"R2 = {tree_r2:.2f}: explains {tree_r2:.0%} of the variation.")

train_r2 = r2_score(y_train, reg_search.predict(X_train))
print(f"Train R2 = {train_r2:.2f} vs test R2 = {tree_r2:.2f}. "
      + ("A large gap means overfitting." if train_r2 - tree_r2 > 0.2 else "The gap is small."))

pred_tree = reg_search.predict(X_test)
mae_ha = mean_absolute_error(np.expm1(y_test), np.expm1(pred_tree))
print(f"MAE in real units: about {mae_ha:.1f} hectares (after undoing log1p).")

## A9. Predicted vs actual

**What this does:** plots each test prediction against the true value. Points on the red diagonal are perfect predictions.

**How to read it:** a flat horizontal band means the model predicts nearly the same value for everything, which is what a weak model on this data looks like.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, pred_tree, alpha=0.6)
lims = [0, max(y_test.max(), pred_tree.max())]
plt.plot(lims, lims, "r--")
plt.xlabel("Actual log_area")
plt.ylabel("Predicted log_area")
plt.title("Predicted vs Actual (regression)")
plt.show()

## A10. Tree plot and feature importance

**What this does:** draws the top 3 levels of the best tree and shows which features it relies on most.

**Key points:**
- Use `best_estimator_` (the tree), not the search object.
- `max_depth=3` limits only the *drawing*, not the model.
- Each box shows the split rule (true goes left), the error, the number of samples, and the average target in that node.
- Importance shows how much each feature reduced error across the tree. It describes the model, it does not prove cause.

In [ ]:
best_reg = reg_search.best_estimator_

plt.figure(figsize=(20, 10))
plot_tree(best_reg, feature_names=list(X_train.columns),
          filled=True, rounded=True, max_depth=3, fontsize=9)
plt.show()

imp_reg = pd.Series(best_reg.feature_importances_, index=X_train.columns).sort_values()
imp_reg.plot(kind="barh", figsize=(8, 6), title="Feature importance (forest fires)")
plt.show()

top3 = imp_reg.sort_values(ascending=False).head(3)
print("FINDING: top 3 features:", ", ".join(f"{k} ({v:.2f})" for k, v in top3.items()))
print(f"Tree depth = {best_reg.get_depth()}, leaves = {best_reg.get_n_leaves()}")

### Part A: summary of findings

Fill in your own numbers from the outputs above.

- **EDA:** peak months were `[months]`. `[x]%` of fires had zero area, so a log transform was needed.
- **Correlation:** the strongest link to `area` was only `[value]`, which means weak linear signal.
- **Model:** the tuned tree had RMSE `[value]` vs baseline `[value]` and R² `[value]`.
- **Interpretation:** top features were `[features]`.
- **Limitations:** about 513 rows, one split, a noisy target, and no fuel, terrain or human-cause data. Do not claim cause from these patterns.
- **Next steps:** repeated CV, Random Forest or Gradient Boosting, or a classification version (large vs small fire).

# Part B: Obesity Levels (Classification)

**Problem:** classify a person into one of 7 weight categories from eating habits, activity, transport and age.

## B1. Load and inspect

**Important context:** about 77% of the rows were generated with SMOTE, so scores may be optimistic for real people.

In [ ]:
url_ob = ("https://archive.ics.uci.edu/static/public/544/"
          "estimation+of+obesity+levels+based+on+eating+habits+and+physical+condition.zip")
habit = pd.read_csv(url_ob, compression="zip")

print("Shape:", habit.shape)
print("Missing values:", habit.isnull().sum().sum())
print("Duplicate rows:", habit.duplicated().sum())
habit.head()

## B2. Clean and check the target

**What this does:** removes duplicates, shows the class counts, and checks how many values look synthetic (non-integers in columns that should be whole numbers).

**Why:** a clear class imbalance means accuracy can mislead, and many non-integer answers confirm the synthetic origin.

In [ ]:
habit = habit.drop_duplicates().reset_index(drop=True)
print(habit.shape)
print(habit["NObeyesdad"].value_counts())

for col in ["Age", "FCVC", "NCP", "CH2O", "FAF", "TUE"]:
    print(f"{col}: {(habit[col] % 1 != 0).mean():.0%} non-integer values")

## B3. Leakage check

**The problem:** the label is defined from BMI, and BMI = Weight / Height². A tree given `Height` and `Weight` can read the label almost directly, so accuracy looks great but says little about lifestyle.

**Switch:** `DROP_LEAKY = True` removes them (honest model). Set it to `False` to see how much the score inflates. Reporting both is a strong addition to the write-up.

In [ ]:
DROP_LEAKY = True

bmi = habit["Weight"] / habit["Height"] ** 2
plt.figure(figsize=(9, 4))
sns.boxplot(x=bmi, y=habit["NObeyesdad"])
plt.title("BMI by class: almost perfectly separated")
plt.show()

## B4. Preprocessing and stratified split

**What this does:**
- `get_dummies` one-hot encodes the nominal features. `LabelEncoder` would invent an order (for example car = 2, bike = 1) that does not exist.
- `LabelEncoder` is used only on the *target*, and we keep it to recover class names for plots.
- `stratify` keeps class proportions the same in train and test.

In [ ]:
leaky_cols = ["Height", "Weight"] if DROP_LEAKY else []
X_c = pd.get_dummies(habit.drop(columns=["NObeyesdad"] + leaky_cols), drop_first=True)

le_target = LabelEncoder()
y_c = le_target.fit_transform(habit["NObeyesdad"])

xtrain, xtest, ytrain, ytest = train_test_split(
    X_c, y_c, test_size=0.2, random_state=42, stratify=y_c
)
print(xtrain.shape, xtest.shape)

## B5. Tune the classification tree

**What this does:** the same randomized search as Part A, but scored with `f1_macro`, which averages the F1 of every class equally. Accuracy can look fine while a small class is predicted badly.

In [ ]:
clf_search = RandomizedSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_dist, n_iter=50, cv=5,
    scoring="f1_macro", random_state=42,
)
clf_search.fit(xtrain, ytrain)
print("Best params:", clf_search.best_params_)
print("Best CV macro F1:", round(clf_search.best_score_, 3))

## B6. Evaluate against a baseline

**What this does:** compares the tree with a `DummyClassifier` that always predicts the most frequent class, then prints a per-class report.

**Reading the report:**
- **Precision:** of the people predicted in a class, how many truly belong.
- **Recall:** of the people truly in a class, how many were found.
- **F1:** balance of the two.

In [ ]:
base_clf = DummyClassifier(strategy="most_frequent").fit(xtrain, ytrain)
pred_base = base_clf.predict(xtest)
pred_clf = clf_search.predict(xtest)

acc_base = accuracy_score(ytest, pred_base)
acc_tree = accuracy_score(ytest, pred_clf)
f1_tree = f1_score(ytest, pred_clf, average="macro")
print(f"Baseline accuracy: {acc_base:.3f}")
print(f"Tree accuracy:     {acc_tree:.3f}")
print(f"Tree macro F1:     {f1_tree:.3f}\n")
print(classification_report(ytest, pred_clf, target_names=le_target.classes_))

## B7. Confusion matrix

**How to read it:** rows are the true class, columns are the prediction, and the diagonal is correct. Errors next to the diagonal mean neighbouring classes get confused, which is expected because the class boundaries are BMI cut-offs.

In [ ]:
cm = confusion_matrix(ytest, pred_clf)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=le_target.classes_, yticklabels=le_target.classes_)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion matrix (obesity)")
plt.xticks(rotation=45, ha="right")
plt.show()

### Findings feedback: classification (automatic)

In [ ]:
print(f"Leakage columns dropped: {DROP_LEAKY}")
if not DROP_LEAKY:
    print("WARNING: Height and Weight are included, so accuracy is inflated by leakage.")

print(f"Tree accuracy {acc_tree:.2f} vs baseline {acc_base:.2f} "
      f"(+{(acc_tree - acc_base) * 100:.0f} points).")

train_acc = accuracy_score(ytrain, clf_search.predict(xtrain))
print(f"Train accuracy {train_acc:.2f} vs test accuracy {acc_tree:.2f}. "
      + ("Large gap: the tree is overfitting." if train_acc - acc_tree > 0.1 else "The gap is small."))

cm_off = cm.copy()
np.fill_diagonal(cm_off, 0)
i, j = np.unravel_index(cm_off.argmax(), cm_off.shape)
print(f"Most common mistake: '{le_target.classes_[i]}' predicted as "
      f"'{le_target.classes_[j]}' ({cm_off[i, j]} times).")

per_class = f1_score(ytest, pred_clf, average=None)
worst = per_class.argmin()
print(f"Weakest class: '{le_target.classes_[worst]}' (F1 = {per_class[worst]:.2f}).")

if DROP_LEAKY and acc_tree > 0.9:
    print("NOTE: very high accuracy even without Height/Weight. Remember ~77% of rows are synthetic, "
          "so near-copies of training rows can appear in the test set and inflate the score.")

## B8. Tree plot and feature importance

**What this does:** draws the top 3 levels (rules a person can read) and lists the most important features.

**How to use it:** the root split is the single most useful separator. Compare the top features with your EDA hypotheses, and treat them as patterns, not causes.

In [ ]:
best_clf = clf_search.best_estimator_

plt.figure(figsize=(20, 10))
plot_tree(best_clf,
          feature_names=list(xtrain.columns),
          class_names=[str(c) for c in le_target.classes_],
          filled=True, rounded=True, max_depth=3, fontsize=9)
# plt.savefig("obesity_tree.png", dpi=300, bbox_inches="tight")   # uncomment to save for GitHub
plt.show()

imp_clf = pd.Series(best_clf.feature_importances_, index=xtrain.columns).sort_values()
imp_clf.tail(10).plot(kind="barh", figsize=(8, 5), title="Top 10 features (obesity)")
plt.show()

top3c = imp_clf.sort_values(ascending=False).head(3)
print("FINDING: top 3 features:", ", ".join(f"{k} ({v:.2f})" for k, v in top3c.items()))
print(f"Tree depth = {best_clf.get_depth()}, leaves = {best_clf.get_n_leaves()}")

### Part B: summary of findings

Fill in your own numbers from the outputs above.

- **Data:** `[n]` rows after removing `[k]` duplicates, about 77% synthetic.
- **Leakage:** with `Height` and `Weight` the accuracy was `[value]`, without them `[value]`.
- **Model:** accuracy `[value]` vs baseline `[value]`, macro F1 `[value]`.
- **Errors:** the most confused classes were `[pair]`, which are neighbouring categories.
- **Interpretation:** top features were `[features]`, and the root split was `[feature <= value]`.
- **Limitations:** synthetic data, one split, and an unstable single tree.
- **Next steps:** repeated stratified CV, Random Forest, and `class_weight="balanced"`.

# Final conclusions

| Question | Forest fires | Obesity |
|---|---|---|
| Did the tree beat the baseline? | `[yes / about equal / no]` | `[yes / no]` |
| Main metric | RMSE `[ ]`, R² `[ ]` | Accuracy `[ ]`, macro F1 `[ ]` |
| Biggest risk | Noisy target, small data | Leakage and synthetic rows |
| Top features | `[ ]` | `[ ]` |

**General lessons:** clean duplicates before splitting, transform skewed targets, always compare with a baseline, remove leaky features, tune with cross-validation, and describe patterns rather than claiming causes.